# NLP - Métricas textuais

## Carregamento dos arquivos

In [1]:
import os
path = "documents"
files_list = [os.path.join(path, file_name) for file_name in os.listdir(path)]

print("Lista de arquivos:", files_list)

Lista de arquivos: ['documents\\An_Ensemble_of_LLMs_Finetuned.pdf', 'documents\\Aroeira_A_Curated_Corpus.pdf', 'documents\\Assessing European and Brazilian Portuguese LLMs for NER in Specialised Domains.pdf', 'documents\\Developing Resource-Efficient Clinical LLMs for Brazilian Portuguese.pdf', 'documents\\ERASMO_Leveraging Large Language Models for Enhanced Clustering Segmentation.pdf', 'documents\\Evaluating Large Language Models for Tax Law Reasoning.pdf', 'documents\\GovBERT-BR_A BERT-Based Language Model for Brazilian Portuguese Governmental Data.pdf', 'documents\\Improving LLMs Reasoning and Planning with Finite-State Machines.pdf', 'documents\\Improving LLMs’ Reasoning and Planning with Finite-State Machines.pdf', 'documents\\LLM-Driven Chest X-Ray Report Generation With a Modular, Reduced-Size Architecture.pdf', 'documents\\Unsupervised Statistical Keyword Extraction Pipeline - Is LLM All You.pdf']


## Definição das funções

In [2]:
# !python -m spacy download en_core_web_sm
# !pip install pymupdf spacy chardet


In [3]:
import os
import fitz  # PyMuPDF
import spacy
from collections import Counter
import chardet

# Carrega o modelo de NLP para português
nlp = spacy.load("en_core_web_sm")

def extract_text_from_pdf(file_path):
    text = ""
    try:
        with fitz.open(file_path) as pdf:
            for page in pdf:
                text += page.get_text()
    except Exception as e:
        print(f"Erro ao ler PDF {file_path}: {e}")
    return text

def extract_title_from_path(file_path):
    filename = os.path.basename(file_path)  # 'An_Agentic_Approach_For_...pdf'
    title_with_underscores = os.path.splitext(filename)[0]  # remove .pdf
    title = title_with_underscores.replace("_", " ")  # substitui _ por espaço
    return title

def remove_references(text):
    keywords = ["Referências", "References"]
    last_pos = -1
    for word in keywords:
        pos = text.rfind(word)
        if pos > last_pos:
            last_pos = pos
    if last_pos != -1:
        return text[:last_pos]
    return text

def load_documents(files):
    return [extract_text_from_pdf(file_path) for file_path in files]

def remove_stopwords(texts):
    cleaned_texts = []
    for text in texts:
        doc = nlp(text)
        tokens = [token.text for token in doc if not token.is_stop]
        cleaned_texts.append(" ".join(tokens))
    return cleaned_texts

def count_sentences(doc):
    return len(list(doc.sents))

def count_tokens(doc):
    return [token.text.lower() for token in doc if token.is_alpha]

def count_pos_tags(doc):
    # Substantivos
    num_nouns = sum(1 for token in doc if token.pos_ == "NOUN")
    # Verbos
    num_verbs = sum(1 for token in doc if token.pos_ == "VERB")
    # Preposições
    num_adpositions = sum(1 for token in doc if token.pos_ == "ADP")
    return num_nouns, num_verbs, num_adpositions

def get_lemmas(doc):
    tokens = [token for token in doc if not token.is_space]
    return [token.lemma_ for token in tokens]

def compute_token_stats(tokens, num_docs):
    total_tokens = len(tokens)
    avg_tokens = total_tokens / num_docs if num_docs else 0
    token_freq = Counter(tokens)
    top_10 = token_freq.most_common(10)
    down_10 = token_freq.most_common()[-10:]
    return total_tokens, avg_tokens, top_10, down_10

def extract_dependencies(doc):
    dependencies = []
    for token in doc:
        dependencies.append(token.dep_)
    return dependencies
# def extract_dependencies(doc):
#     dependencies = []
#     for token in doc:
#         dependencies.append({
#             "text": token.text,
#             "head": token.head.text,
#             "dep": token.dep_,
#             "pos": token.pos_
#         })
#     return dependencies

def get_doc_statistics(texts):
    total_sentences = 0
    total_tokens_list = []
    total_nouns = total_verbs = total_preps = 0

    for text in texts:
        doc = nlp(text)

        total_sentences += count_sentences(doc)

        tokens = count_tokens(doc)
        total_tokens_list.extend(tokens)

        nouns, verbs, preps = count_pos_tags(doc)
        total_nouns += nouns
        total_verbs += verbs
        total_preps += preps

    num_docs = len(texts)
    avg_sentences = total_sentences / num_docs if num_docs else 0
    total_tokens, avg_tokens, top_10, down_10 = compute_token_stats(total_tokens_list, num_docs)
    dependencies = extract_dependencies(doc)
    lemmas = get_lemmas(doc)

    return {
        "num_sentences": total_sentences,
        "avg_sentences_per_doc": avg_sentences,
        "num_tokens": total_tokens,
        "avg_tokens_per_doc": avg_tokens,
        "top_10_tokens": top_10,
        "down_10_tokens": down_10,
        "num_nouns": total_nouns,
        "num_verbs": total_verbs,
        "num_prepositions": total_preps,
        "dependencies": dependencies,
        "lemmas": lemmas
    }
    

c:\Users\rafae\AppData\Local\Programs\Python\Python311\Lib\site-packages\thinc\compat.py:36: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  hasattr(torch, "has_mps")
c:\Users\rafae\AppData\Local\Programs\Python\Python311\Lib\site-packages\thinc\compat.py:37: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  and torch.has_mps  # type: ignore[attr-defined]


## Estatísticas

### Sem remoção de stopwords

In [4]:
texts = load_documents(files_list)
texts = [remove_references(text) for text in texts]
stats = get_doc_statistics(texts)

# Exibe as estatísticas
for key, value in stats.items():
    print(f"{key}: {value}")

num_sentences: 2776
avg_sentences_per_doc: 252.36363636363637
num_tokens: 49797
avg_tokens_per_doc: 4527.0
top_10_tokens: [('the', 2915), ('and', 1549), ('of', 1401), ('to', 1112), ('in', 986), ('a', 926), ('for', 679), ('is', 542), ('models', 468), ('this', 447)]
down_10_tokens: [('guaranteed', 1), ('needing', 1), ('ﬂow', 1), ('demanding', 1), ('gpus', 1), ('fore', 1), ('depends', 1), ('captured', 1), ('contextualized', 1), ('ﬁnanced', 1)]
num_nouns: 14752
num_verbs: 6237
num_prepositions: 5770
dependencies: ['amod', 'compound', 'compound', 'dep', 'compound', 'nsubj', 'punct', 'ROOT', 'attr', 'attr', 'nsubj', 'dep', 'relcl', 'punct', 'dep', 'compound', 'compound', 'appos', 'punct', 'punct', 'compound', 'conj', 'punct', 'cc', 'compound', 'nmod', 'compound', 'compound', 'dep', 'conj', 'prep', 'pobj', 'punct', 'compound', 'conj', 'prep', 'compound', 'pobj', 'punct', 'compound', 'conj', 'punct', 'compound', 'dep', 'conj', 'punct', 'dep', 'punct', 'compound', 'dep', 'appos', 'punct', 'comp

In [5]:
import pandas as pd
df_stats = pd.DataFrame([stats])
df_stats

,num_sentences,avg_sentences_per_doc,num_tokens,avg_tokens_per_doc,top_10_tokens,down_10_tokens,num_nouns,num_verbs,num_prepositions,dependencies,lemmas
0,2776,252.363636,49797,4527.0,"[(the, 2915), (and, 1549), (of, 1401), (to, 11...","[(guaranteed, 1), (needing, 1), (ﬂow, 1), (dem...",14752,6237,5770,"[amod, compound, compound, dep, compound, nsub...","[Unsupervised, Statistical, Keyword, Extractio..."


### Com remoção de stopwords

In [6]:
texts = load_documents(files_list)
texts = remove_stopwords(texts)
stats = get_doc_statistics(texts)

# Exibe as estatísticas
for key, value in stats.items():
    print(f"{key}: {value}")

num_sentences: 4145
avg_sentences_per_doc: 376.8181818181818
num_tokens: 35931
avg_tokens_per_doc: 3266.4545454545455
top_10_tokens: [('models', 570), ('language', 366), ('model', 315), ('text', 253), ('al', 242), ('et', 239), ('legal', 235), ('data', 217), ('portuguese', 207), ('llms', 205)]
down_10_tokens: [('oshiro', 1), ('mafra', 1), ('roller', 1), ('rose', 1), ('engel', 1), ('cramer', 1), ('cowley', 1), ('vehicle', 1), ('veh', 1), ('technol', 1)]
num_nouns: 15897
num_verbs: 6202
num_prepositions: 133
dependencies: ['compound', 'compound', 'compound', 'dep', 'compound', 'dep', 'punct', 'nsubj', 'dep', 'ROOT', 'punct', 'dep', 'compound', 'compound', 'npadvmod', 'punct', 'punct', 'compound', 'nsubj', 'punct', 'nmod', 'nmod', 'compound', 'compound', 'dep', 'compound', 'appos', 'punct', 'compound', 'compound', 'compound', 'conj', 'punct', 'compound', 'nsubj', 'punct', 'conj', 'dep', 'ROOT', 'punct', 'dep', 'punct', 'compound', 'dep', 'dobj', 'punct', 'nsubj', 'nmod', 'amod', 'compound'

In [7]:
import pandas as pd
df_stats = pd.DataFrame([stats])
df_stats

,num_sentences,avg_sentences_per_doc,num_tokens,avg_tokens_per_doc,top_10_tokens,down_10_tokens,num_nouns,num_verbs,num_prepositions,dependencies,lemmas
0,4145,376.818182,35931,3266.454545,"[(models, 570), (language, 366), (model, 315),...","[(oshiro, 1), (mafra, 1), (roller, 1), (rose, ...",15897,6202,133,"[compound, compound, compound, dep, compound, ...","[Unsupervised, Statistical, Keyword, Extractio..."


# Montagem JSON

In [8]:
from langdetect import detect  # para identificar o idioma automaticamente

def build_article_json(file_path, text, stats):
    # Detecta idioma do texto
    try:
        idioma = detect(text)
        idioma = "Português" if idioma == "pt" else "Inglês"
    except:
        idioma = "Desconhecido"
        
    # Lista de tokens e respectivas tags
    doc = nlp(text)
    tokens = [token.text for token in doc if token.is_alpha]
    pos_tags = [token.pos_ for token in doc if token.is_alpha]
    lemmas = [token.lemma_ for token in doc if token.is_alpha]
    deps = [token.dep_ for token in doc if token.is_alpha]

    # Cria o JSON do artigo
    article_json = {
        "titulo": extract_title_from_path(file_path),
        "informacoes_url": "http://exemplo.com/artigo",  # placeholder
        "idioma": idioma,
        "storage_key": file_path,
        "autores": [  # você pode tentar extrair isso com heurísticas, se quiser
            {
                "nome": "Autor Desconhecido",
                "afiliacao": "Instituição Desconhecida",
                "orcid": "http://orcid.org/0000-0000-0000-0000"
            }
        ],
        "data_publicacao": "xx/xx/xxxx",
        "resumo": "Resumo não disponível",
        "keywords": [],
        "referencias": [],  # se quiser, podemos implementar extração de referências
        "artigo_completo_PT": text if idioma == "Português" else "",
        "artigo_completo_EN": text if idioma == "Inglês" else "",
        "artigo_tokenizado": tokens,
        "pos_tagger": pos_tags,
        "lema": lemmas,
        "dep": deps
    }
    return article_json


# Processo principal
texts = load_documents(files_list)
texts = [remove_references(text) for text in texts]

all_articles_json = []
for file_path, text in zip(files_list, texts):
    stats = get_doc_statistics([text])
    article_json = build_article_json(file_path, text, stats)
    all_articles_json.append(article_json)

# Exibe como JSON
import json
print(json.dumps(all_articles_json, indent=2, ensure_ascii=False))


[
  {
    "titulo": "An Ensemble of LLMs Finetuned",
    "informacoes_url": "http://exemplo.com/artigo",
    "idioma": "Inglês",
    "storage_key": "documents\\An_Ensemble_of_LLMs_Finetuned.pdf",
    "autores": [
      {
        "nome": "Autor Desconhecido",
        "afiliacao": "Instituição Desconhecida",
        "orcid": "http://orcid.org/0000-0000-0000-0000"
      }
    ],
    "data_publicacao": "xx/xx/xxxx",
    "resumo": "Resumo não disponível",
    "keywords": [],
    "referencias": [],
    "artigo_completo_PT": "",
    "artigo_completo_EN": "An Ensemble of LLMs Finetuned\nwith LoRA for NER in Portuguese Legal\nDocuments\nRafael Oleques Nunes(B)\n, Letícia Maria Puttlitz\n, Antonio Oss Boll\n,\nAndre Spritzer\n, Carla Maria Dal Sasso Freitas\n, Dennis Giovani Balreira\n,\nand Anderson Rocha Tavares\nFederal University of Rio Grande do Sul, Porto Alegre, Brazil\n{ronunes,spritzer,carla,dgbalreira,artavares}@inf.ufrgs.br,\n{leticia.puttlitz,antonio.boll}@ufrgs.br\nAbstract. Given t

In [9]:
import json

with open("corpus.json", "w", encoding="utf-8") as f:
    json.dump(all_articles_json, f, ensure_ascii=False, indent=2)


In [ ]:
api_key = "api_key_here"  # Substitua pela sua chave de API do OpenAI
from openai import OpenAI

# Initialize client
client = OpenAI(api_key=api_key)

def translate_english_to_portuguese(text):
    prompt = f"Translate the following text from English to Portuguese:\n\nEnglish: {text}\nPortuguese:"
    try:
        response = client.chat.completions.create(
            model="o4-mini",  # Or "gpt-3.5-turbo", "gpt-4", etc.
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        translation = response.choices[0].message.content.strip()
        return translation
    except Exception as e:
        print(f"Translation failed for: {text[:60]}... \nError: {e}")
        return ""

# Example: Loop through articles safely
for article in all_articles_json:
    en_text = article.get("artigo_completo_EN", "")
    if en_text:
        pt_text = translate_english_to_portuguese(en_text)
        article["artigo_completo_PT"] = pt_text

In [37]:
import json

with open("corpus.json", "w", encoding="utf-8") as f:
    json.dump(all_articles_json, f, ensure_ascii=False, indent=2)


In [ ]:
# !pip install openai==0.28

In [38]:
# Find articles with failed (empty) translations
failed_translations = [
    article for article in all_articles_json
    if article.get("artigo_completo_PT", "") == ""
]

# Print summary or inspect them
print(f"Total failed translations: {len(failed_translations)}")


Total failed translations: 0
